# VisualTorque × Qwen-27B Recipient Recode Dose-Response Confirmation

The unit-dose experiment produced large, direction-specific margin improvements but no categorical answer flips.

[实验设置与运行说明](../../../docs/experiment_notes/05_appendix_content_and_encoding/2748c5673bb51c59.md)


In [ ]:
# Run once. Every later code cell reports both cell and inner-loop progress.
import re
import threading
import time
from IPython.core.magic import register_cell_magic
from tqdm.auto import tqdm

_TRACKED_CELLS = 17
_SETUP_PROGRESS = tqdm(total=1, desc='Initialize progress display', unit='cell')
_ACTIVE_CELL_PROGRESS = None

def cell_stage(message):
    if _ACTIVE_CELL_PROGRESS is not None:
        _ACTIVE_CELL_PROGRESS['stage'] = str(message)
        elapsed = time.monotonic() - _ACTIVE_CELL_PROGRESS['started']
        _ACTIVE_CELL_PROGRESS['bar'].set_postfix_str(
            f"{message} | elapsed {elapsed:.0f}s", refresh=True
        )

@register_cell_magic
def cell_progress(line, cell):
    global _ACTIVE_CELL_PROGRESS
    specification = line.strip() or 'Cell progress'
    match = re.fullmatch(r'(\d+)/(\d+)\s+(.+)', specification)
    if match:
        current, total, label = int(match.group(1)), int(match.group(2)), match.group(3)
        initial = current - 1
    else:
        current, total, label, initial = 1, 1, specification, 0
    started = time.monotonic()
    bar = tqdm(total=total, initial=initial, desc=label, unit='cell',
               dynamic_ncols=True, leave=True)
    state = {'bar': bar, 'started': started, 'stage': 'starting'}
    _ACTIVE_CELL_PROGRESS = state
    stopped = threading.Event()
    def heartbeat():
        while not stopped.wait(1.0):
            elapsed = time.monotonic() - started
            bar.set_postfix_str(
                f"{state['stage']} | elapsed {elapsed:.0f}s", refresh=True
            )
    worker = threading.Thread(target=heartbeat, daemon=True)
    worker.start()
    succeeded = False
    try:
        cell_stage('running')
        result = get_ipython().run_cell(cell)
        result.raise_error()
        succeeded = True
        return result.result
    finally:
        stopped.set(); worker.join(timeout=2.0)
        elapsed = time.monotonic() - started
        if succeeded:
            bar.n = current
            bar.set_postfix_str(f"complete | elapsed {elapsed:.0f}s", refresh=True)
        else:
            bar.set_postfix_str(f"failed | elapsed {elapsed:.0f}s", refresh=True)
        bar.close(); _ACTIVE_CELL_PROGRESS = None

_SETUP_PROGRESS.update(1); _SETUP_PROGRESS.close()
print(f'Cell progress helper ready: {_TRACKED_CELLS} tracked cells.')


In [ ]:
%%cell_progress 1/17 Mount Drive and locate VisualTorque
cell_stage('Mounting Google Drive')
from google.colab import drive
drive.mount('/content/drive')

cell_stage('Locating VisualTorque project')
from pathlib import Path
PROJECT_CANDIDATES = (
    Path('/content/drive/MyDrive/VisualTorque'),
    Path('/content/drive/MyDrive/VisualTorque/VisualTorque'),
)
PROJECT_ROOT = next(
    (
        path for path in PROJECT_CANDIDATES
        if (path / 'toolkit' / 'pyproject.toml').is_file()
    ),
    None,
)
assert PROJECT_ROOT is not None, (
    'Could not find VisualTorque/toolkit/pyproject.toml on Drive.'
)
TOOLKIT_ROOT = PROJECT_ROOT / 'toolkit'
print('Project:', PROJECT_ROOT)


In [ ]:
%%cell_progress 2/17 Install and verify the multimodal environment
import importlib
import subprocess
import sys

cell_stage('Removing unused TorchAudio')
subprocess.run(
    [sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchaudio'],
    check=False,
)
cell_stage('Installing pinned inference and analysis dependencies')
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q', '-U',
    'transformers==5.14.0', 'accelerate>=1.10', 'sentencepiece',
    'protobuf', 'huggingface_hub>=0.34', 'tqdm>=4.66',
    'pandas>=2.2', 'matplotlib>=3.8', 'scikit-learn>=1.5',
])

cell_stage('Registering the live VisualTorque source tree')
SRC_ROOT = TOOLKIT_ROOT / 'src'
PACKAGE_ROOT = SRC_ROOT / 'visual_torque_toolkit'
assert (PACKAGE_ROOT / '__init__.py').is_file(), PACKAGE_ROOT
source_text = str(SRC_ROOT)
sys.path[:] = [entry for entry in sys.path if entry != source_text]
sys.path.insert(0, source_text)
for module_name in tuple(sys.modules):
    if (
        module_name == 'visual_torque_toolkit'
        or module_name.startswith('visual_torque_toolkit.')
    ):
        del sys.modules[module_name]
importlib.invalidate_caches()

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torchvision
import visual_torque_toolkit
from IPython.display import display
from transformers import AutoModelForMultimodalLM, AutoProcessor
from visual_torque_toolkit.behavior.adapter import HuggingFaceMultimodalAdapter

required_features = (
    '_template', '_to_device', '_candidate_plan', '_append_text_prefix',
)
assert all(
    hasattr(HuggingFaceMultimodalAdapter, name)
    for name in required_features
)
toolkit_path = Path(visual_torque_toolkit.__file__).resolve()
assert PACKAGE_ROOT.resolve() in toolkit_path.parents, toolkit_path
print('torch:', torch.__version__, '| CUDA:', torch.version.cuda)
print('torchvision:', torchvision.__version__)
print('Toolkit source:', toolkit_path)


In [ ]:
%%cell_progress 3/17 Verify GPU and Hugging Face access
cell_stage('Checking CUDA device')
assert torch.cuda.is_available(), 'Select an A100 GPU runtime.'
print(torch.cuda.get_device_name(0))
print('CUDA memory (GB):', round(
    torch.cuda.get_device_properties(0).total_memory / 2**30, 1
))

cell_stage('Authenticating with Hugging Face')
from google.colab import userdata
from huggingface_hub import login
try:
    token = userdata.get('HF_TOKEN')
    if token:
        login(token=token, add_to_git_credential=False)
        print('Hugging Face login complete.')
    else:
        print('HF_TOKEN is not set. Public checkpoints may still work.')
except Exception as exc:
    print('Hugging Face login skipped:', exc)


In [ ]:
%%cell_progress 4/17 Freeze the dose-response confirmation protocol
cell_stage('Locking doses, routes, controls, discovery selection, and confirmation tests')
import hashlib
import json
from datetime import datetime, timezone

MODEL_ID = 'Qwen/Qwen3.5-27B'
MODEL_REVISION = '373612cb4f9288fcf550b526d86d85febedd9801'
MODEL_DTYPE = 'bfloat16'
DEVICE_MAP = 'auto'
EXPECTED_LAYER_COUNT = 64
INTERVENTION_BATCH_SIZE = 4
IMAGE_TOKEN_ID_OVERRIDE = None
UPSTREAM_SCOPE = 'options_tokens'
SEED = 20261206
STAGE_SITES = (
    {'stage': 'physical_handoff', 'layer_index': 49, 'scope': 'options_tokens'},
    {'stage': 'response_code', 'layer_index': 55, 'scope': 'final_prompt_token'},
)
CAPTURE_LAYERS = tuple(site['layer_index'] for site in STAGE_SITES)
STAGE_BY_NAME = {site['stage']: site for site in STAGE_SITES}
DOSES = (0.0, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0)
LINEAR_DOSES = (0.0, 0.5, 1.0, 1.5, 2.0)
ROUTES = ('direct', 'sequential')
DOSE_MODES = (
    'recoded', 'target_code_only', 'random_recoded', 'reverse_target'
)
BASELINE_CONDITIONS = (
    'clean_target_base', 'natural_target_edited',
    'direct_raw', 'sequential_total',
)
RECORDS_PER_PAIR = len(BASELINE_CONDITIONS) + (
    len(DOSES) * len(ROUTES) * len(DOSE_MODES)
)
DISCOVERY_SUCCESS_THRESHOLD = 0.50
MINIMUM_CONFIRMATION_PAIRS = 5
MINIMUM_CONFIRMATION_FAMILIES = 4
BOOTSTRAP_REPLICATES = 10000
ALPHA_LEVEL = 0.05
SMOKE_CENTERED_LOGPROB_TOLERANCE = 0.15
NORM_RELATIVE_TOLERANCE = 1e-3

PARENT_ROOT = (
    PROJECT_ROOT / 'results'
    / 'qwen27b_recipient_conditioned_semantic_recode_rescue_v1'
)
PARENT_RUN_PATH = PARENT_ROOT / 'RUN_COMPLETE.json'
PARENT_DECISION_PATH = PARENT_ROOT / 'analysis' / 'recipient_recode_decision.json'
PARENT_PAIR_PATH = PARENT_ROOT / 'mapping_pair_manifest.jsonl'
PARENT_RECORDS_PATH = PARENT_ROOT / 'analysis' / 'condition_records.csv'
PARENT_FACTOR_ROOT = (
    PROJECT_ROOT / 'results'
    / 'qwen27b_outcomeflip_withincontext_route_confirmation_v2_expanded'
)
PARENT_FACTOR_PATH = PARENT_FACTOR_ROOT / 'confirmation_manifest.jsonl'

AXIS_ROOT = PROJECT_ROOT / 'results' / 'qwen27b_symmetric_code_axis_v1'
AXIS_PROTOCOL_PATH = AXIS_ROOT / 'frozen_protocol.json'
AXIS_LIBRARY_PATH = AXIS_ROOT / 'direction_library.json'
AXIS_TENSOR_PATH = AXIS_ROOT / 'direction_tensors.pt'

OUTPUT_ROOT = (
    PROJECT_ROOT / 'results'
    / 'qwen27b_recipient_recode_dose_response_confirmation_v1'
)
PROTOCOL_LOCK_PATH = OUTPUT_ROOT / 'frozen_protocol.json'
PAIR_MANIFEST_PATH = OUTPUT_ROOT / 'frozen_pair_manifest.jsonl'
SHARD_DIR = OUTPUT_ROOT / 'dose_shards'
ANALYSIS_DIR = OUTPUT_ROOT / 'analysis'
FIGURE_DIR = ANALYSIS_DIR / 'figures'
DOSE_SELECTION_PATH = ANALYSIS_DIR / 'discovery_dose_selection.json'
for directory in (OUTPUT_ROOT, SHARD_DIR, ANALYSIS_DIR, FIGURE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

def json_default(value):
    if isinstance(value, np.generic): return value.item()
    if isinstance(value, Path): return str(value)
    if torch.is_tensor(value) and value.numel() == 1: return value.item()
    raise TypeError(type(value).__name__)

def write_json_atomic(file_path, value):
    file_path = Path(file_path)
    temporary = file_path.with_suffix(file_path.suffix + '.tmp')
    temporary.write_text(
        json.dumps(value, indent=2, ensure_ascii=False,
                   default=json_default) + '\n', encoding='utf-8'
    )
    temporary.replace(file_path)

def write_jsonl_atomic(file_path, rows):
    file_path = Path(file_path)
    temporary = file_path.with_suffix(file_path.suffix + '.tmp')
    with temporary.open('w', encoding='utf-8') as handle:
        for row in rows:
            handle.write(json.dumps(
                row, sort_keys=True, default=json_default
            ) + '\n')
    temporary.replace(file_path)

def read_jsonl(file_path):
    with Path(file_path).open('r', encoding='utf-8') as handle:
        return [json.loads(line) for line in handle if line.strip()]

def sha256_file(file_path):
    digest = hashlib.sha256()
    with Path(file_path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return 'sha256:' + digest.hexdigest()

def stable_key(value):
    return hashlib.sha256(f'{SEED}:{value!r}'.encode('utf-8')).hexdigest()

print('Frozen doses:', DOSES)
print('Records per pair:', RECORDS_PER_PAIR)
print('Output:', OUTPUT_ROOT)


In [ ]:
%%cell_progress 5/17 Prepare Dataset v1 on local SSD
cell_stage('Checking local and Drive dataset caches')
from zipfile import ZipFile

REQUIRED_DATASET_FILES = {
    'dataset_config.json', 'scenes.jsonl', 'counterfactual_pairs.jsonl',
    'questions.jsonl', 'splits.json', 'statistics.json',
}
DATASET_DIR = Path('/content/visualtorque_dataset_v1')
DATASET_ARCHIVE = PROJECT_ROOT / 'data' / 'dataset_v1.zip'
LEGACY_DATASET_DIR = PROJECT_ROOT / 'data' / 'dataset_v1'

def dataset_complete(path):
    return path.is_dir() and all(
        (path / name).is_file() for name in REQUIRED_DATASET_FILES
    )

if not dataset_complete(DATASET_DIR):
    DATASET_DIR.mkdir(parents=True, exist_ok=True)
    if DATASET_ARCHIVE.is_file():
        cell_stage('Extracting Dataset v1 archive')
        with ZipFile(DATASET_ARCHIVE, 'r') as bundle:
            members = [m for m in bundle.infolist() if not m.is_dir()]
            root = DATASET_DIR.resolve()
            for member in tqdm(members, desc='Extracting Dataset v1', unit='file'):
                target = (DATASET_DIR / member.filename).resolve()
                if root not in target.parents:
                    raise ValueError(f'Unsafe archive path: {member.filename}')
                bundle.extract(member, DATASET_DIR)
    elif dataset_complete(LEGACY_DATASET_DIR):
        DATASET_DIR = LEGACY_DATASET_DIR
    else:
        raise FileNotFoundError(
            'Run the Dataset v1 build/cache cell in the behavior notebook first.'
        )
assert dataset_complete(DATASET_DIR)
print('Dataset:', DATASET_DIR)


In [ ]:
%%cell_progress 6/17 Load the frozen pairs and independent response axes
cell_stage('Verifying the subthreshold parent result and preserving its split')
from collections import Counter, defaultdict
from visual_torque_toolkit.behavior import (
    BehaviorRunConfig, PromptCondition, load_behavior_examples,
)

for required in (
    PARENT_RUN_PATH, PARENT_DECISION_PATH, PARENT_PAIR_PATH,
    PARENT_RECORDS_PATH, PARENT_FACTOR_PATH,
    AXIS_PROTOCOL_PATH, AXIS_LIBRARY_PATH, AXIS_TENSOR_PATH,
):
    assert Path(required).exists(), f'Missing prerequisite: {required}'

parent_run = json.loads(PARENT_RUN_PATH.read_text(encoding='utf-8'))
parent_decision = json.loads(PARENT_DECISION_PATH.read_text(encoding='utf-8'))
assert parent_run['primary_decision'] == 'recipient_conditioned_recode_not_supported'
assert parent_decision['direct_recode_gate']
assert parent_decision['sequential_recode_gate']
assert not parent_decision['copy_reduction_gate']

pair_manifest = read_jsonl(PARENT_PAIR_PATH)
pair_manifest.sort(key=lambda row: row['pair_id'])
assert len(pair_manifest) == int(parent_run['pairs'])
assert all(row['edge_type'] == 'mapping' for row in pair_manifest)
assert all(row['reference_mapping'] != row['target_mapping'] for row in pair_manifest)
assert all(row['reference_orientation'] == row['target_orientation'] for row in pair_manifest)
assert all(
    row['reference_edited_label'] != row['target_edited_label']
    for row in pair_manifest
)

factor_manifest = read_jsonl(PARENT_FACTOR_PATH)
row_by_family = {str(row['family_id']): row for row in factor_manifest}
assert not ({row['family_id'] for row in pair_manifest} - set(row_by_family))

axis_metadata = json.loads(AXIS_LIBRARY_PATH.read_text(encoding='utf-8'))
axis_payload = torch.load(AXIS_TENSOR_PATH, map_location='cpu', weights_only=False)
assert axis_payload['model_id'] == MODEL_ID
assert axis_payload['revision'] == MODEL_REVISION
RESPONSE_AXES = {
    key: value.float() for key, value in axis_payload['symmetric_axes'].items()
}
CONFIRMATION_AXES = {
    key: value.float() for key, value in axis_payload['confirmation_axes'].items()
}
assert set(RESPONSE_AXES) == set(CONFIRMATION_AXES) == {'AB', 'AC', 'BC'}

protocol_identity = {
    'schema_version': 'recipient-recode-dose-response-confirmation-1.0',
    'analysis_status': 'discovery_selected_dose_with_heldout_confirmation',
    'model_id': MODEL_ID, 'revision': MODEL_REVISION,
    'parent_run_sha256': sha256_file(PARENT_RUN_PATH),
    'parent_pair_sha256': sha256_file(PARENT_PAIR_PATH),
    'axis_tensor_sha256': sha256_file(AXIS_TENSOR_PATH),
    'pair_ids_and_splits': [
        [row['pair_id'], row['family_id'], row['split']]
        for row in pair_manifest
    ],
    'doses': list(DOSES), 'linear_doses': list(LINEAR_DOSES),
    'routes': list(ROUTES), 'modes': list(DOSE_MODES),
    'discovery_selection_rule': (
        'per route choose the smallest dose with discovery target success '
        'at least 0.50 and nonnegative mean desired-vs-source margin; if '
        'none, choose the dose with lexicographically greatest success then '
        'margin, breaking ties toward the smaller dose'
    ),
    'confirmation_primary': (
        'selected target recode versus random, reverse, raw/total, and '
        'target-code-only; no confirmation outcome enters dose selection'
    ),
    'confirmation_reuse_boundary': (
        'the parent split is preserved and confirmation rows are held out '
        'from current dose selection, but the same families were previously '
        'observed at the unit dose'
    ),
    'local_slope_range': list(LINEAR_DOSES),
    'behavioral_rescue_rule': (
        'confirmation target success at least 0.50 and at least 0.20 above '
        'both random and reverse controls'
    ),
    'physical_residual_rule': (
        'recoded margin exceeds target-code-only margin with family-bootstrap '
        '95% interval above zero'
    ),
    'directions_fit_on_current_pairs': False,
    'bootstrap_unit': 'complete physical family',
    'seed': SEED,
}
if PROTOCOL_LOCK_PATH.is_file():
    previous = json.loads(PROTOCOL_LOCK_PATH.read_text(encoding='utf-8'))
    assert previous.get('identity') == protocol_identity, (
        'Existing protocol differs. Use a new versioned output directory.'
    )
else:
    write_json_atomic(PROTOCOL_LOCK_PATH, {
        'identity': protocol_identity,
        'locked_at_utc': datetime.now(timezone.utc).isoformat(),
    })
write_jsonl_atomic(PAIR_MANIFEST_PATH, pair_manifest)

loader_config = BehaviorRunConfig(
    dataset_dir=DATASET_DIR, output_dir=OUTPUT_ROOT / '_unused_loader',
    model_id=MODEL_ID, model_revision=MODEL_REVISION,
    conditions=(PromptCondition.IMAGE_RULE,), question_kinds=('outcome',),
    splits=('probe_train', 'probe_dev', 'probe_test'), batch_size=1,
    generation_every=0, seed=SEED,
)
outcome_examples = tuple(load_behavior_examples(loader_config))
outcome_by_scene = {}
for example in outcome_examples:
    outcome_by_scene.setdefault(str(example.scene_id), example)
required_scene_ids = {
    str(role['scene_id'])
    for row in factor_manifest for roles in row['orientation_roles'].values()
    for role in roles.values()
}
assert not (required_scene_ids - set(outcome_by_scene))

support_table = pd.DataFrame(pair_manifest).groupby(
    ['split', 'stale_error']
).size().rename('pairs')
print('Frozen pairs / families:',
      len(pair_manifest), len({row['family_id'] for row in pair_manifest}))
display(support_table)


## Frozen L49 physical handoff and L55 response-code sites


In [ ]:
%%cell_progress 7/17 Load pinned Qwen-27B and capture stages
cell_stage('Loading Qwen-27B and resolving the two frozen layers')
import gc
if 'adapter' in globals():
    del adapter
    gc.collect()
    torch.cuda.empty_cache()

adapter = HuggingFaceMultimodalAdapter(
    MODEL_ID, revision=MODEL_REVISION,
    dtype=MODEL_DTYPE, device_map=DEVICE_MAP,
)
display(adapter.metadata)

def resolve_module_path(root, module_path):
    value = root
    for component in module_path.split('.'):
        value = getattr(value, component)
    return value

def discover_decoder_layers(model):
    candidates = (
        'model.language_model.layers', 'model.text_model.layers',
        'language_model.model.layers', 'model.model.layers', 'model.layers',
    )
    for module_path in candidates:
        try:
            layers = resolve_module_path(model, module_path)
        except AttributeError:
            continue
        if isinstance(layers, torch.nn.ModuleList) and len(layers) >= 8:
            return module_path, layers
    raise RuntimeError('Could not locate decoder layers.')

LAYER_PATH, DECODER_LAYERS = discover_decoder_layers(adapter.model)
assert len(DECODER_LAYERS) == EXPECTED_LAYER_COUNT
CAPTURE_MODULES = {
    layer_index: DECODER_LAYERS[layer_index]
    for layer_index in CAPTURE_LAYERS
}
print('Layer path:', LAYER_PATH)
print('Capture layers:', CAPTURE_LAYERS)


In [ ]:
%%cell_progress 8/17 Define paired prompts, labels, and exact masks
cell_stage('Defining answer mappings and label-aware scoring')
import math
from dataclasses import replace as dataclass_replace

RULE_TEXT = (
    'Each square is one unit of weight. Distance is measured in tick '
    'intervals from the center pivot. Torque equals weight times distance. '
    'Total torque on a side is the sum over stacks on that side.'
)
QUESTION_TEXT = 'Which way will the balance scale tip?'
SEMANTIC_TEXT = {
    'left': 'left side', 'balanced': 'equal', 'right': 'right side',
}

def mapping_values(mapping_variant):
    semantics = (
        ('left', 'balanced', 'right')
        if mapping_variant == 'forward'
        else ('right', 'balanced', 'left')
    )
    return tuple(zip(('A', 'B', 'C'), semantics))

def custom_prompt(mapping):
    options = '\n'.join(
        f'{label}. {SEMANTIC_TEXT[semantic]}'
        for label, semantic in mapping
    )
    return '\n\n'.join([
        'Answer this question using only the supplied balance-scale scene.',
        RULE_TEXT,
        'Question: ' + QUESTION_TEXT,
        'Options:\n' + options,
        'Do not explain your reasoning. Reply with exactly one option '
        'letter and nothing else. Valid replies are: A, B, C.',
    ])

def role_record(row, orientation, role_name):
    return row['orientation_roles'][orientation][role_name]

def changed_side_for(row, orientation):
    return row['orientation_changed_side'][orientation]

def make_example(row, orientation, mapping_variant, role_name):
    mapping = mapping_values(mapping_variant)
    role = role_record(row, orientation, role_name)
    scene_id = str(role['scene_id'])
    semantics = tuple(value for _, value in mapping)
    labels = tuple(label for label, _ in mapping)
    outcome = str(role['outcome'])
    base = outcome_by_scene[scene_id]
    return dataclass_replace(
        base,
        example_id=(
            f"weight_factorial:{row['family_id']}:{orientation}:"
            f"{mapping_variant}:{role_name}"
        ),
        question_id=(
            f"weight_factorial_{row['family_id']}_{orientation}_{mapping_variant}"
        ),
        question_kind='outcome', variant='weight_mapping_side_factorial',
        prompt=custom_prompt(mapping), labels=labels,
        option_text=tuple(SEMANTIC_TEXT[value] for value in semantics),
        option_semantics=semantics,
        gold_label=labels[semantics.index(outcome)], gold_semantic=outcome,
    )

def output_hidden_tensor(output):
    if torch.is_tensor(output):
        hidden = output
    elif isinstance(output, (tuple, list)) and output and torch.is_tensor(output[0]):
        hidden = output[0]
    elif hasattr(output, 'last_hidden_state'):
        hidden = output.last_hidden_state
    else:
        raise TypeError(type(output).__name__)
    assert hidden.ndim == 3
    return hidden

def replace_output_hidden(output, modified):
    if torch.is_tensor(output):
        return modified
    if isinstance(output, tuple):
        return (modified,) + output[1:]
    if isinstance(output, list):
        return [modified] + output[1:]
    raise TypeError(type(output).__name__)

def prepare_examples(examples):
    examples = tuple(examples)
    plan = adapter._candidate_plan(examples[0])
    if plan is None:
        raise RuntimeError('A/B/C labels must be single-token candidates.')
    for example in examples[1:]:
        assert adapter._candidate_plan(example) == plan
    fixed_prefix, token_by_label = plan
    inputs = adapter._append_text_prefix(
        adapter._to_device(adapter._template(
            [example.messages() for example in examples],
            add_generation_prompt=True,
        )),
        fixed_prefix,
    )
    return {'examples': examples, 'inputs': inputs, 'token_by_label': token_by_label}

def discover_masks(prepared):
    assert len(prepared['examples']) == 1
    example = prepared['examples'][0]
    ids = prepared['inputs']['input_ids'][0].detach().cpu()
    attention = prepared['inputs']['attention_mask'][0].detach().cpu().bool()
    tokenizer = getattr(adapter.processor, 'tokenizer', adapter.processor)
    tokens = tokenizer.convert_ids_to_tokens(ids.tolist())
    image_ids = set()
    if IMAGE_TOKEN_ID_OVERRIDE is not None:
        values = IMAGE_TOKEN_ID_OVERRIDE if isinstance(
            IMAGE_TOKEN_ID_OVERRIDE, (list, tuple)
        ) else [IMAGE_TOKEN_ID_OVERRIDE]
        image_ids.update(int(value) for value in values)
    for token_id, token_text in zip(ids.tolist(), tokens):
        lower = str(token_text).lower()
        if any(marker in lower for marker in ('image', 'vision', '<img', 'img>')):
            image_ids.add(int(token_id))
    image = torch.tensor(
        [int(token_id) in image_ids for token_id in ids.tolist()], dtype=torch.bool
    ) & attention
    assert bool(image.any()), 'No image tokens detected.'
    final = torch.zeros_like(attention)
    final[int(torch.where(attention)[0][-1])] = True
    all_text = attention & ~image & ~final

    encoding = tokenizer(
        example.prompt, add_special_tokens=False, return_offsets_mapping=True,
    )
    prompt_ids = list(encoding['input_ids'])
    offsets = list(encoding['offset_mapping'])
    id_list = ids.tolist()
    starts = [
        start for start in range(len(ids) - len(prompt_ids) + 1)
        if id_list[start:start + len(prompt_ids)] == prompt_ids
    ]
    assert len(starts) == 1
    prompt_start = starts[0]
    sections = example.prompt.split('\n\n')
    names = (
        'intro_tokens', 'rule_tokens', 'question_tokens',
        'options_tokens', 'answer_instruction_tokens',
    )
    assert len(sections) == len(names)
    masks = {
        'valid_tokens': attention, 'image_tokens': image,
        'all_text_except_final': all_text, 'final_prompt_token': final,
    }
    spans = []
    cursor = 0
    for name, section in zip(names, sections):
        start = example.prompt.find(section, cursor)
        end = start + len(section)
        cursor = end
        spans.append((name, start, end))
        masks[name] = torch.zeros_like(attention)
    for offset_index, (token_start, token_end) in enumerate(offsets):
        overlap, name = max(
            (max(0, min(token_end, end) - max(token_start, start)), name)
            for name, start, end in spans
        )
        full_index = prompt_start + offset_index
        if overlap > 0 and bool(all_text[full_index]):
            masks[name][full_index] = True
    covered = torch.zeros_like(attention)
    for name in names:
        covered |= masks[name]
    masks['separator_chat_tokens'] = all_text & ~covered
    assert torch.equal(
        masks['options_tokens'] | masks['intro_tokens'] | masks['rule_tokens']
        | masks['question_tokens'] | masks['answer_instruction_tokens']
        | masks['separator_chat_tokens'], all_text,
    )
    masks['tokens'] = tokens
    return masks

def assert_aligned(prepared_by_name):
    names = list(prepared_by_name)
    reference = prepared_by_name[names[0]]
    reference_masks = discover_masks(reference)
    for name in names[1:]:
        current = prepared_by_name[name]
        for key in ('input_ids', 'attention_mask'):
            assert torch.equal(reference['inputs'][key], current['inputs'][key])
        masks = discover_masks(current)
        for scope in (UPSTREAM_SCOPE, 'final_prompt_token'):
            assert torch.equal(reference_masks[scope], masks[scope])
    return reference_masks

def score_records(prepared, logits):
    records = []
    for batch_index, example in enumerate(prepared['examples']):
        logprobs = torch.log_softmax(logits[batch_index, -1, :].float(), dim=-1)
        scores = [
            float(logprobs[prepared['token_by_label'][label]].detach().cpu())
            for label in example.labels
        ]
        maximum = max(scores)
        weights = [math.exp(value - maximum) for value in scores]
        predicted_index = int(np.argmax(scores))
        records.append({
            'scene_id': str(example.scene_id),
            'example_id': str(example.example_id),
            'candidate_semantics': list(example.option_semantics),
            'candidate_labels': list(example.labels),
            'candidate_logprobs': scores,
            'candidate_probabilities': [value / sum(weights) for value in weights],
            'predicted_semantic': example.option_semantics[predicted_index],
            'predicted_label': example.labels[predicted_index],
            'gold_semantic': example.gold_semantic,
            'gold_label': example.gold_label,
        })
    return records

def centered_error(first, second):
    first = np.asarray(first, dtype=float)
    second = np.asarray(second, dtype=float)
    return float(np.max(np.abs(
        (first - first.mean()) - (second - second.mean())
    )))

def forward_score_prepared(prepared):
    with torch.inference_mode():
        output = adapter.model(**prepared['inputs'], use_cache=False)
    return score_records(prepared, output.logits)

print('Fully paired prompts and masks ready.')


In [ ]:
%%cell_progress 9/17 Define stage capture and reciprocal interventions
cell_stage('Defining layerwise update transport with matched controls')

def forward_capture(prepared):
    assert len(prepared['examples']) == 1
    captured = {}
    handles = []
    for layer_index, module in CAPTURE_MODULES.items():
        def capture(_module, _inputs, output, layer_index=layer_index):
            captured[layer_index] = (
                output_hidden_tensor(output)[0].detach().cpu().to(torch.bfloat16)
            )
        handles.append(module.register_forward_hook(capture))
    try:
        with torch.inference_mode():
            output = adapter.model(**prepared['inputs'], use_cache=False)
    finally:
        for handle in handles: handle.remove()
    assert set(captured) == set(CAPTURE_LAYERS)
    return score_records(prepared, output.logits)[0], captured

def forward_score(example):
    prepared = prepare_examples([example])
    return forward_score_prepared(prepared)[0]

def centered_scores(record):
    values = np.asarray(record['candidate_logprobs'], dtype=float)
    return values - values.mean()

def centered_error(first, second):
    first = np.asarray(first, dtype=float); second = np.asarray(second, dtype=float)
    return float(np.max(np.abs(
        (first - first.mean()) - (second - second.mean())
    )))

def masked_delta(changed, base, mask):
    result = torch.zeros_like(changed, dtype=torch.bfloat16)
    selected = mask.bool()
    result[selected] = (
        changed[selected].float() - base[selected].float()
    ).to(torch.bfloat16)
    return result

def selected_norm(tensor, mask):
    return float(torch.linalg.vector_norm(tensor[mask.bool()].float()))

def match_norm(tensor, reference, mask):
    result = torch.zeros_like(tensor, dtype=torch.float32)
    selected = mask.bool(); values = tensor[selected].float()
    current = torch.linalg.vector_norm(values)
    target = torch.linalg.vector_norm(reference[selected].float())
    if float(current) > 0 and float(target) > 0:
        values = values * (target / current)
    else:
        values.zero_()
    result[selected] = values
    return result.to(torch.bfloat16)

def random_like(reference, mask, key):
    generator = torch.Generator(device='cpu')
    generator.manual_seed(int(stable_key(key)[:16], 16) % (2**63 - 1))
    result = torch.zeros_like(reference, dtype=torch.float32)
    selected = mask.bool()
    values = torch.randn(
        reference[selected].shape, generator=generator, dtype=torch.float32
    )
    target = torch.linalg.vector_norm(reference[selected].float())
    values = values * (target / torch.linalg.vector_norm(values))
    result[selected] = values
    return result.to(torch.bfloat16)

def build_pair_examples(pair):
    row = row_by_family[pair['family_id']]
    base_role = f"target_{pair['target_key']}_base"
    edited_role = f"target_{pair['target_key']}_hybrid"
    return {
        'reference_base': make_example(
            row, pair['reference_orientation'], pair['reference_mapping'], base_role
        ),
        'reference_edited': make_example(
            row, pair['reference_orientation'], pair['reference_mapping'], edited_role
        ),
        'target_base': make_example(
            row, pair['target_orientation'], pair['target_mapping'], base_role
        ),
        'target_edited': make_example(
            row, pair['target_orientation'], pair['target_mapping'], edited_role
        ),
    }

def prepare_pair(pair):
    examples = build_pair_examples(pair)
    prepared = {name: prepare_examples([example]) for name, example in examples.items()}
    masks = {name: discover_masks(value) for name, value in prepared.items()}
    shapes = {tuple(value['inputs']['input_ids'].shape) for value in prepared.values()}
    assert len(shapes) == 1
    for site in STAGE_SITES:
        scope = site['scope']
        reference = masks['target_base'][scope]
        assert all(torch.equal(reference, current[scope]) for current in masks.values())
        assert bool(reference.any())
    return examples, prepared, masks

def score_stage_conditions(example, stage, conditions, mask_override=None):
    results = []
    layer_index = int(stage['layer_index'])
    for start in range(0, len(conditions), INTERVENTION_BATCH_SIZE):
        actual = list(conditions[start:start + INTERVENTION_BATCH_SIZE])
        padded = actual + [actual[-1]] * (INTERVENTION_BATCH_SIZE - len(actual))
        prepared = prepare_examples([example] * INTERVENTION_BATCH_SIZE)
        single = prepare_examples([example])
        masks = discover_masks(single)
        selected_cpu = (
            mask_override.bool() if mask_override is not None
            else masks[stage['scope']].bool()
        )
        def hook(_module, _inputs, output):
            hidden = output_hidden_tensor(output)
            selected = selected_cpu.to(hidden.device)
            modified = hidden.clone()
            for batch_index, condition in enumerate(padded):
                payload = condition['payload'].to(hidden.device, hidden.dtype)
                if condition['mode'] == 'add':
                    modified[batch_index, selected] = (
                        hidden[batch_index, selected] + payload[selected]
                    )
                elif condition['mode'] == 'replace':
                    modified[batch_index, selected] = payload[selected]
                else:
                    raise ValueError(condition['mode'])
            return replace_output_hidden(output, modified)
        handle = DECODER_LAYERS[layer_index].register_forward_hook(hook)
        try:
            with torch.inference_mode():
                output = adapter.model(**prepared['inputs'], use_cache=False)
        finally:
            handle.remove()
        current = score_records(prepared, output.logits)[:len(actual)]
        for record, condition in zip(current, actual):
            record['condition'] = condition['condition']
            record['payload_norm'] = selected_norm(
                condition['payload'], selected_cpu
            )
            results.append(record)
    return results

def counterfactual_progress(record, base, edited):
    current = centered_scores(record); start = centered_scores(base)
    destination = centered_scores(edited); direction = destination - start
    denominator = float(direction @ direction)
    assert denominator > 1e-12
    return float((current - start) @ direction / denominator)

def label_margin(record, positive_label, negative_label):
    labels = list(record['candidate_labels'])
    values = np.asarray(record['candidate_logprobs'], dtype=float)
    return float(values[labels.index(positive_label)] - values[labels.index(negative_label)])

print('Stage capture and reciprocal intervention helpers ready.')


In [ ]:
%%cell_progress 10/17 Define batched direct and sequential dose interventions
cell_stage('Defining direction-specific dose paths without tuning on confirmation')

L49_STAGE = STAGE_BY_NAME['physical_handoff']
L55_STAGE = STAGE_BY_NAME['response_code']

def signed_response_axis(start_label, end_label, confirmation=False):
    start_label, end_label = str(start_label), str(end_label)
    assert start_label in 'ABC' and end_label in 'ABC' and start_label != end_label
    pair = ''.join(sorted((start_label, end_label)))
    library = CONFIRMATION_AXES if confirmation else RESPONSE_AXES
    axis = library[pair].float()
    assert axis.ndim == 2 and axis.shape[0] == 1
    return axis if start_label < end_label else -axis

def unit_axis(tensor):
    tensor = tensor.float()
    norm = torch.linalg.vector_norm(tensor)
    assert float(norm) > 0
    return tensor / norm

def orthogonal_random_axis(source_axis, target_axis, key):
    generator = torch.Generator(device='cpu')
    generator.manual_seed(int(stable_key(key)[:16], 16) % (2**63 - 1))
    value = torch.randn(source_axis.shape, generator=generator, dtype=torch.float32)
    basis = []
    for candidate in (source_axis, target_axis):
        candidate = candidate.float().clone()
        for item in basis:
            candidate = candidate - torch.sum(candidate * item) * item
        if float(torch.linalg.vector_norm(candidate)) > 1e-4:
            basis.append(unit_axis(candidate))
    for item in basis:
        value = value - torch.sum(value * item) * item
    return unit_axis(value)

def selected_payload(template, mask, selected):
    result = torch.zeros_like(template, dtype=torch.bfloat16)
    assert int(mask.sum()) == int(selected.shape[0])
    result[mask.bool()] = selected.to(torch.bfloat16)
    return result

def decompose_source_update(source_update, source_axis):
    source_update = source_update.float()
    source_axis = unit_axis(source_axis)
    coefficient = torch.sum(source_update * source_axis)
    component = coefficient * source_axis
    residual = source_update - component
    return float(coefficient), component, residual

def alpha_key(alpha):
    return str(float(alpha)).replace('.', 'p')

def dose_condition(route, mode, alpha):
    return f'{route}__{mode}__a{alpha_key(alpha)}'

def score_sequential_dose_conditions(
    example, l49_payload, clean_l55_selected,
    source_axis, target_axis, random_axis, source_strength, conditions,
):
    records = []
    single = prepare_examples([example])
    masks = discover_masks(single)
    l49_mask = masks[L49_STAGE['scope']].bool()
    l55_mask = masks[L55_STAGE['scope']].bool()
    for start in range(0, len(conditions), INTERVENTION_BATCH_SIZE):
        actual = list(conditions[start:start + INTERVENTION_BATCH_SIZE])
        padded = actual + [actual[-1]] * (INTERVENTION_BATCH_SIZE - len(actual))
        prepared = prepare_examples([example] * INTERVENTION_BATCH_SIZE)
        measurements = []

        def l49_hook(_module, _inputs, output):
            hidden = output_hidden_tensor(output)
            selected = l49_mask.to(hidden.device)
            payload = l49_payload.to(hidden.device, hidden.dtype)
            modified = hidden.clone()
            for batch_index in range(INTERVENTION_BATCH_SIZE):
                modified[batch_index, selected] = (
                    hidden[batch_index, selected] + payload[selected]
                )
            return replace_output_hidden(output, modified)

        def l55_hook(_module, _inputs, output):
            hidden = output_hidden_tensor(output)
            selected = l55_mask.to(hidden.device)
            clean = clean_l55_selected.to(hidden.device, torch.float32)
            source = unit_axis(source_axis).to(hidden.device)
            target = unit_axis(target_axis).to(hidden.device)
            random = unit_axis(random_axis).to(hidden.device)
            modified = hidden.clone()
            measurements.clear()
            for batch_index, condition in enumerate(padded):
                current = hidden[batch_index, selected].float()
                delta = current - clean
                current_source = torch.sum(delta * source)
                without_source = current - current_source * source
                alpha = float(condition['alpha'])
                mode = condition['mode']
                if mode == 'total':
                    replacement = current
                elif mode == 'recoded':
                    replacement = without_source + alpha * float(source_strength) * target
                elif mode == 'target_code_only':
                    replacement = clean + alpha * float(source_strength) * target
                elif mode == 'random_recoded':
                    replacement = without_source + alpha * float(source_strength) * random
                elif mode == 'reverse_target':
                    replacement = without_source - alpha * float(source_strength) * target
                else:
                    raise ValueError(mode)
                modified[batch_index, selected] = replacement.to(hidden.dtype)
                measurements.append({
                    'current_source_coefficient': float(current_source.detach().cpu()),
                    'pre_recode_l55_delta_norm': float(
                        torch.linalg.vector_norm(delta).detach().cpu()
                    ),
                })
            return replace_output_hidden(output, modified)

        handles = [
            DECODER_LAYERS[L49_STAGE['layer_index']].register_forward_hook(l49_hook),
            DECODER_LAYERS[L55_STAGE['layer_index']].register_forward_hook(l55_hook),
        ]
        try:
            with torch.inference_mode():
                output = adapter.model(**prepared['inputs'], use_cache=False)
        finally:
            for handle in handles: handle.remove()
        current_records = score_records(prepared, output.logits)[:len(actual)]
        for record, condition, measurement in zip(
            current_records, actual, measurements[:len(actual)]
        ):
            record.update(condition)
            record.update(measurement)
            record['patched_tokens_l49'] = int(l49_mask.sum())
            record['patched_tokens_l55'] = int(l55_mask.sum())
            records.append(record)
    return records

def decorate_record(record, pair, condition, route, mode, alpha,
                    scores, source_coefficient, component_fraction):
    desired_label = scores['target_edited']['gold_label']
    source_label = scores['reference_edited']['gold_label']
    error_label = scores['target_edited']['predicted_label']
    record.update({
        'pair_id': pair['pair_id'], 'family_id': pair['family_id'],
        'split': pair['split'], 'stale_error': pair['stale_error'],
        'condition': condition, 'route': route, 'mode': mode,
        'alpha': float(alpha), 'desired_label': desired_label,
        'source_label': source_label, 'error_label': error_label,
        'source_coefficient': float(source_coefficient),
        'source_component_fraction': float(component_fraction),
        'desired_vs_source_margin': label_margin(
            record, desired_label, source_label
        ),
        'desired_vs_error_margin': label_margin(
            record, desired_label, error_label
        ),
        'reaches_desired_label': bool(record['predicted_label'] == desired_label),
        'copies_source_label': bool(record['predicted_label'] == source_label),
    })
    return record

print('Dose intervention helpers ready.')


In [ ]:
%%cell_progress 11/17 Run dose-zero parity and anti-copy smoke tests
cell_stage('Checking reconstruction, alpha-zero equivalence, and code geometry')

smoke_pair = pair_manifest[0]
smoke_examples, smoke_prepared, smoke_masks = prepare_pair(smoke_pair)
smoke_scores, smoke_hidden = {}, {}
for name, prepared in smoke_prepared.items():
    smoke_scores[name], smoke_hidden[name] = forward_capture(prepared)

assert smoke_scores['reference_edited']['gold_label'] != smoke_scores['target_edited']['gold_label']
l55_mask = smoke_masks['target_base']['final_prompt_token']
target_l55 = masked_delta(
    smoke_hidden['target_edited'][55], smoke_hidden['target_base'][55], l55_mask
)
reference_l55 = masked_delta(
    smoke_hidden['reference_edited'][55], smoke_hidden['reference_base'][55], l55_mask
)
reference_l55 = match_norm(reference_l55, target_l55, l55_mask)
source_axis = signed_response_axis(
    smoke_scores['reference_base']['gold_label'],
    smoke_scores['reference_edited']['gold_label'],
)
target_axis = signed_response_axis(
    smoke_scores['target_base']['gold_label'],
    smoke_scores['target_edited']['gold_label'],
)
random_axis = orthogonal_random_axis(
    source_axis, target_axis, ('smoke-dose-random', smoke_pair['pair_id'])
)
coefficient, component, residual = decompose_source_update(
    reference_l55[l55_mask], source_axis
)
assert float(torch.max(torch.abs(
    reference_l55[l55_mask].float() - component - residual
))) < 1e-5
assert abs(float(torch.sum(random_axis * source_axis))) < 1e-4
assert abs(float(torch.sum(random_axis * target_axis))) < 1e-4

zero_payload = selected_payload(reference_l55, l55_mask, residual)
zero_records = score_stage_conditions(
    smoke_examples['target_base'], L55_STAGE,
    [
        {'condition': name, 'payload': zero_payload, 'mode': 'add'}
        for name in ('recoded_a0', 'random_a0', 'reverse_a0')
    ],
)
zero_error = max(
    centered_error(
        zero_records[0]['candidate_logprobs'],
        record['candidate_logprobs'],
    ) for record in zero_records[1:]
)
assert zero_error < SMOKE_CENTERED_LOGPROB_TOLERANCE, zero_error
print('Smoke pair:', smoke_pair['pair_id'])
print('Alpha-zero maximum centered difference:', zero_error)
print('Source coefficient:', coefficient)


## Prospective dose grid and held-out confirmation


In [ ]:
%%cell_progress 12/17 Run the frozen direct and sequential dose grid
cell_stage('Capturing resumable pair shards across all doses and controls')

capture_identity = {
    'protocol_sha256': sha256_file(PROTOCOL_LOCK_PATH),
    'pair_manifest_sha256': sha256_file(PAIR_MANIFEST_PATH),
    'axis_tensor_sha256': sha256_file(AXIS_TENSOR_PATH),
    'model_id': MODEL_ID, 'revision': MODEL_REVISION,
    'doses': list(DOSES), 'routes': list(ROUTES),
    'modes': list(DOSE_MODES), 'records_per_pair': RECORDS_PER_PAIR,
}
CAPTURE_SHA256 = 'sha256:' + hashlib.sha256(
    json.dumps(capture_identity, sort_keys=True).encode('utf-8')
).hexdigest()

for pair in tqdm(pair_manifest, desc='Recipient recode dose grid', unit='pair'):
    shard_path = SHARD_DIR / f"{pair['pair_id']}.json"
    if shard_path.is_file():
        existing = json.loads(shard_path.read_text(encoding='utf-8'))
        assert existing['capture_identity_sha256'] == CAPTURE_SHA256
        assert len(existing['records']) == RECORDS_PER_PAIR
        continue

    examples, prepared, masks = prepare_pair(pair)
    scores, hidden = {}, {}
    for name, current in prepared.items():
        scores[name], hidden[name] = forward_capture(current)
    assert scores['reference_base']['predicted_label'] == scores['reference_base']['gold_label']
    assert scores['reference_edited']['predicted_label'] == scores['reference_edited']['gold_label']
    assert scores['target_base']['predicted_label'] == scores['target_base']['gold_label']
    assert scores['target_edited']['predicted_label'] != scores['target_edited']['gold_label']

    l49_mask = masks['target_base'][L49_STAGE['scope']]
    l55_mask = masks['target_base'][L55_STAGE['scope']]
    reference_l49 = masked_delta(
        hidden['reference_edited'][49], hidden['reference_base'][49], l49_mask
    )
    target_l49 = masked_delta(
        hidden['target_edited'][49], hidden['target_base'][49], l49_mask
    )
    reference_l55 = masked_delta(
        hidden['reference_edited'][55], hidden['reference_base'][55], l55_mask
    )
    target_l55 = masked_delta(
        hidden['target_edited'][55], hidden['target_base'][55], l55_mask
    )
    reference_l49 = match_norm(reference_l49, target_l49, l49_mask)
    reference_l55 = match_norm(reference_l55, target_l55, l55_mask)

    source_axis = signed_response_axis(
        scores['reference_base']['gold_label'],
        scores['reference_edited']['gold_label'],
    )
    target_axis = signed_response_axis(
        scores['target_base']['gold_label'],
        scores['target_edited']['gold_label'],
    )
    random_axis = orthogonal_random_axis(
        source_axis, target_axis, ('dose-random', pair['pair_id'])
    )
    source_coefficient, source_component, source_residual = decompose_source_update(
        reference_l55[l55_mask], source_axis
    )
    source_norm = float(torch.linalg.vector_norm(reference_l55[l55_mask].float()))
    component_fraction = float(
        torch.linalg.vector_norm(source_component) / max(source_norm, 1e-12)
    )

    records = []
    for condition, baseline, route, mode in (
        ('clean_target_base', dict(scores['target_base']), 'baseline', 'clean'),
        ('natural_target_edited', dict(scores['target_edited']), 'baseline', 'natural'),
    ):
        records.append(decorate_record(
            baseline, pair, condition, route, mode, -1.0,
            scores, source_coefficient, component_fraction,
        ))

    raw_record = score_stage_conditions(
        examples['target_base'], L55_STAGE, [{
            'condition': 'direct_raw', 'payload': reference_l55, 'mode': 'add',
        }]
    )[0]
    records.append(decorate_record(
        raw_record, pair, 'direct_raw', 'direct', 'raw', -1.0,
        scores, source_coefficient, component_fraction,
    ))

    total_record = score_sequential_dose_conditions(
        examples['target_base'], reference_l49,
        hidden['target_base'][55][l55_mask],
        source_axis, target_axis, random_axis, source_coefficient,
        [{'condition': 'sequential_total', 'route': 'sequential',
          'mode': 'total', 'alpha': -1.0}],
    )[0]
    records.append(decorate_record(
        total_record, pair, 'sequential_total', 'sequential', 'total', -1.0,
        scores, source_coefficient, component_fraction,
    ))

    direct_specs = []
    for alpha in DOSES:
        selected_by_mode = {
            'recoded': source_residual + alpha * source_coefficient * target_axis,
            'target_code_only': alpha * source_coefficient * target_axis,
            'random_recoded': source_residual + alpha * source_coefficient * random_axis,
            'reverse_target': source_residual - alpha * source_coefficient * target_axis,
        }
        for mode in DOSE_MODES:
            direct_specs.append({
                'condition': dose_condition('direct', mode, alpha),
                'route': 'direct', 'mode': mode, 'alpha': float(alpha),
                'payload': selected_payload(
                    reference_l55, l55_mask, selected_by_mode[mode]
                ),
            })
    direct_records = score_stage_conditions(
        examples['target_base'], L55_STAGE,
        [{
            'condition': spec['condition'],
            'payload': spec['payload'], 'mode': 'add',
        } for spec in direct_specs],
    )
    for record, spec in zip(direct_records, direct_specs):
        records.append(decorate_record(
            record, pair, spec['condition'], spec['route'], spec['mode'],
            spec['alpha'], scores, source_coefficient, component_fraction,
        ))

    sequential_specs = [
        {
            'condition': dose_condition('sequential', mode, alpha),
            'route': 'sequential', 'mode': mode, 'alpha': float(alpha),
        }
        for alpha in DOSES for mode in DOSE_MODES
    ]
    sequential_records = score_sequential_dose_conditions(
        examples['target_base'], reference_l49,
        hidden['target_base'][55][l55_mask],
        source_axis, target_axis, random_axis, source_coefficient,
        sequential_specs,
    )
    for record, spec in zip(sequential_records, sequential_specs):
        records.append(decorate_record(
            record, pair, spec['condition'], spec['route'], spec['mode'],
            spec['alpha'], scores, source_coefficient, component_fraction,
        ))

    assert len(records) == RECORDS_PER_PAIR
    assert len({row['condition'] for row in records}) == RECORDS_PER_PAIR
    write_json_atomic(shard_path, {
        'schema_version': 'recipient-recode-dose-pair-shard-1.0',
        'capture_identity_sha256': CAPTURE_SHA256,
        'pair': pair,
        'source_coefficient': source_coefficient,
        'source_component_fraction': component_fraction,
        'records': records,
    })
    del hidden
    torch.cuda.empty_cache()

assert len(list(SHARD_DIR.glob('*.json'))) == len(pair_manifest)
write_json_atomic(OUTPUT_ROOT / 'capture_complete.json', {
    'capture_identity': capture_identity,
    'capture_identity_sha256': CAPTURE_SHA256,
    'pairs': len(pair_manifest),
    'completed_at_utc': datetime.now(timezone.utc).isoformat(),
})
print('Complete dose shards:', len(pair_manifest))


In [ ]:
%%cell_progress 13/17 Release GPU and assemble the dose curves
cell_stage('Freeing model memory and combining all pair shards')
import gc
del adapter
gc.collect()
torch.cuda.empty_cache()

records = []
for path in sorted(SHARD_DIR.glob('*.json')):
    payload = json.loads(path.read_text(encoding='utf-8'))
    assert payload['capture_identity_sha256'] == CAPTURE_SHA256
    assert len(payload['records']) == RECORDS_PER_PAIR
    records.extend(payload['records'])

record_frame = pd.DataFrame(records)
assert len(record_frame) == len(pair_manifest) * RECORDS_PER_PAIR
required_columns = [
    'desired_vs_source_margin', 'desired_vs_error_margin',
    'reaches_desired_label', 'copies_source_label',
    'source_coefficient', 'source_component_fraction',
]
assert not record_frame[required_columns].isna().any().any()
record_frame.to_csv(ANALYSIS_DIR / 'dose_condition_records.csv', index=False)

dose_frame = record_frame[record_frame['alpha'] >= 0].copy()
dose_summary = dose_frame.groupby(
    ['split', 'route', 'mode', 'alpha'], as_index=False
).agg(
    pairs=('pair_id', 'size'),
    families=('family_id', 'nunique'),
    desired_margin=('desired_vs_source_margin', 'mean'),
    desired_success=('reaches_desired_label', 'mean'),
    source_copy=('copies_source_label', 'mean'),
)
dose_summary.to_csv(ANALYSIS_DIR / 'dose_curves.csv', index=False)
print('Records / dose records:', len(record_frame), len(dose_frame))
display(dose_summary[
    (dose_summary['split'] == 'confirmation')
    & (dose_summary['mode'] == 'recoded')
].round(4))


In [ ]:
%%cell_progress 14/17 Select one dose per route on discovery only
cell_stage('Applying the frozen smallest-effective-dose rule')

selection = {}
selection_rows = []
for route in ROUTES:
    candidates = dose_summary[
        (dose_summary['split'] == 'discovery')
        & (dose_summary['route'] == route)
        & (dose_summary['mode'] == 'recoded')
    ].sort_values('alpha').copy()
    assert set(candidates['alpha'].astype(float)) == set(DOSES)
    eligible = candidates[
        (candidates['desired_success'] >= DISCOVERY_SUCCESS_THRESHOLD)
        & (candidates['desired_margin'] >= 0)
    ]
    if not eligible.empty:
        chosen = eligible.sort_values('alpha').iloc[0]
        threshold_met = True
        reason = 'smallest_threshold_crossing'
    else:
        chosen = candidates.sort_values(
            ['desired_success', 'desired_margin', 'alpha'],
            ascending=[False, False, True],
        ).iloc[0]
        threshold_met = False
        reason = 'best_available_fallback'
    selection[route] = float(chosen['alpha'])
    selection_rows.append({
        'route': route, 'selected_alpha': float(chosen['alpha']),
        'discovery_success': float(chosen['desired_success']),
        'discovery_margin': float(chosen['desired_margin']),
        'threshold_met': threshold_met, 'selection_reason': reason,
    })

dose_selection = {
    'schema_version': 'recipient-recode-discovery-dose-selection-1.0',
    'selected_alpha': selection,
    'rows': selection_rows,
    'confirmation_rows_read_during_selection': 0,
    'rule': protocol_identity['discovery_selection_rule'],
}
write_json_atomic(DOSE_SELECTION_PATH, dose_selection)
display(pd.DataFrame(selection_rows))


In [ ]:
%%cell_progress 15/17 Test dose slopes and held-out categorical rescue
cell_stage('Testing only the frozen confirmation pairs at selected doses')

def family_interval(frame, value_column, key):
    family_values = frame.groupby('family_id')[value_column].mean().to_numpy(dtype=float)
    estimate = float(family_values.mean())
    generator = np.random.default_rng(
        int(stable_key(('bootstrap', key))[:16], 16) % (2**63 - 1)
    )
    draws = np.empty(BOOTSTRAP_REPLICATES, dtype=float)
    for index in range(BOOTSTRAP_REPLICATES):
        draws[index] = generator.choice(
            family_values, size=len(family_values), replace=True
        ).mean()
    return {
        'estimate': estimate,
        'ci_low': float(np.quantile(draws, ALPHA_LEVEL / 2)),
        'ci_high': float(np.quantile(draws, 1 - ALPHA_LEVEL / 2)),
        'pairs': int(len(frame)), 'families': int(len(family_values)),
    }

confirmation = dose_frame[dose_frame['split'] == 'confirmation'].copy()
confirmation_baseline = record_frame[
    (record_frame['split'] == 'confirmation')
    & (record_frame['condition'].isin(['direct_raw', 'sequential_total']))
].copy()

selected_rows = []
contrast_rows = []
for route in ROUTES:
    selected_alpha = selection[route]
    current = confirmation[
        (confirmation['route'] == route)
        & np.isclose(confirmation['alpha'], selected_alpha)
    ].copy()
    pivot_margin = current.pivot(
        index=['pair_id', 'family_id'], columns='mode',
        values='desired_vs_source_margin',
    ).reset_index()
    pivot_success = current.pivot(
        index=['pair_id', 'family_id'], columns='mode',
        values='reaches_desired_label',
    ).reset_index()
    baseline_name = 'direct_raw' if route == 'direct' else 'sequential_total'
    baseline = confirmation_baseline[
        confirmation_baseline['condition'] == baseline_name
    ][['pair_id', 'family_id', 'desired_vs_source_margin',
       'reaches_desired_label']].rename(columns={
        'desired_vs_source_margin': 'baseline_margin',
        'reaches_desired_label': 'baseline_success',
    })
    merged_margin = pivot_margin.merge(
        baseline[['pair_id', 'family_id', 'baseline_margin']],
        on=['pair_id', 'family_id'], validate='one_to_one',
    )
    merged_success = pivot_success.merge(
        baseline[['pair_id', 'family_id', 'baseline_success']],
        on=['pair_id', 'family_id'], validate='one_to_one',
    )
    for control in ('random_recoded', 'reverse_target', 'target_code_only'):
        for metric, merged in (
            ('margin', merged_margin), ('success', merged_success)
        ):
            values = merged[['pair_id', 'family_id']].copy()
            values['value'] = (
                merged['recoded'].astype(float) - merged[control].astype(float)
            )
            contrast_rows.append({
                'route': route, 'selected_alpha': selected_alpha,
                'contrast': f'recoded_vs_{control}', 'metric': metric,
                **family_interval(
                    values, 'value', (route, selected_alpha, control, metric)
                ),
            })
    for metric, merged, baseline_column in (
        ('margin', merged_margin, 'baseline_margin'),
        ('success', merged_success, 'baseline_success'),
    ):
        values = merged[['pair_id', 'family_id']].copy()
        values['value'] = (
            merged['recoded'].astype(float) - merged[baseline_column].astype(float)
        )
        contrast_rows.append({
            'route': route, 'selected_alpha': selected_alpha,
            'contrast': 'recoded_vs_baseline', 'metric': metric,
            **family_interval(
                values, 'value', (route, selected_alpha, 'baseline', metric)
            ),
        })
    for mode in DOSE_MODES:
        values = current[current['mode'] == mode].copy()
        selected_rows.append({
            'route': route, 'selected_alpha': selected_alpha, 'mode': mode,
            'mean_margin': float(values['desired_vs_source_margin'].mean()),
            'success_rate': float(values['reaches_desired_label'].mean()),
            'source_copy_rate': float(values['copies_source_label'].mean()),
            'pairs': len(values), 'families': values['family_id'].nunique(),
        })

slope_rows = []
for route in ROUTES:
    local = confirmation[
        (confirmation['route'] == route)
        & (confirmation['alpha'].isin(LINEAR_DOSES))
    ]
    for (pair_id, family_id, mode), current in local.groupby(
        ['pair_id', 'family_id', 'mode']
    ):
        current = current.sort_values('alpha')
        assert tuple(current['alpha'].astype(float)) == LINEAR_DOSES
        slope = float(np.polyfit(
            current['alpha'].to_numpy(dtype=float),
            current['desired_vs_source_margin'].to_numpy(dtype=float), 1,
        )[0])
        slope_rows.append({
            'route': route, 'pair_id': pair_id,
            'family_id': family_id, 'mode': mode, 'slope': slope,
        })
slope_frame = pd.DataFrame(slope_rows)
for route in ROUTES:
    current = slope_frame[slope_frame['route'] == route].pivot(
        index=['pair_id', 'family_id'], columns='mode', values='slope'
    ).reset_index()
    for control in ('random_recoded', 'reverse_target'):
        values = current[['pair_id', 'family_id']].copy()
        values['value'] = current['recoded'] - current[control]
        contrast_rows.append({
            'route': route, 'selected_alpha': selection[route],
            'contrast': f'local_slope_vs_{control}', 'metric': 'margin_slope',
            **family_interval(values, 'value', (route, 'slope', control)),
        })
    values = current[['pair_id', 'family_id']].copy()
    values['value'] = current['recoded']
    contrast_rows.append({
        'route': route, 'selected_alpha': selection[route],
        'contrast': 'local_recoded_slope', 'metric': 'margin_slope',
        **family_interval(values, 'value', (route, 'slope', 'zero')),
    })

selected_frame = pd.DataFrame(selected_rows)
contrast_frame = pd.DataFrame(contrast_rows)
selected_frame.to_csv(ANALYSIS_DIR / 'selected_dose_confirmation_rates.csv', index=False)
contrast_frame.to_csv(ANALYSIS_DIR / 'confirmation_contrasts.csv', index=False)
slope_frame.to_csv(ANALYSIS_DIR / 'confirmation_pair_slopes.csv', index=False)
display(selected_frame.round(4))
display(contrast_frame.round(4))

def lookup(route, contrast, metric):
    return contrast_frame[
        (contrast_frame['route'] == route)
        & (contrast_frame['contrast'] == contrast)
        & (contrast_frame['metric'] == metric)
    ].iloc[0]

def rate(route, mode):
    return float(selected_frame[
        (selected_frame['route'] == route)
        & (selected_frame['mode'] == mode)
    ]['success_rate'].iloc[0])

confirmation_pairs = confirmation['pair_id'].nunique()
confirmation_families = confirmation['family_id'].nunique()
support_gate = bool(
    confirmation_pairs >= MINIMUM_CONFIRMATION_PAIRS
    and confirmation_families >= MINIMUM_CONFIRMATION_FAMILIES
)
route_gates = {}
for route in ROUTES:
    slope_gate = bool(
        lookup(route, 'local_recoded_slope', 'margin_slope')['ci_low'] > 0
        and lookup(route, 'local_slope_vs_random_recoded', 'margin_slope')['ci_low'] > 0
        and lookup(route, 'local_slope_vs_reverse_target', 'margin_slope')['ci_low'] > 0
    )
    specificity_gate = bool(
        lookup(route, 'recoded_vs_random_recoded', 'margin')['ci_low'] > 0
        and lookup(route, 'recoded_vs_reverse_target', 'margin')['ci_low'] > 0
    )
    residual_gate = bool(
        lookup(route, 'recoded_vs_target_code_only', 'margin')['ci_low'] > 0
    )
    rescue_gate = bool(
        rate(route, 'recoded') >= DISCOVERY_SUCCESS_THRESHOLD
        and rate(route, 'recoded') - rate(route, 'random_recoded') >= 0.20
        and rate(route, 'recoded') - rate(route, 'reverse_target') >= 0.20
    )
    route_gates[route] = {
        'slope_gate': slope_gate, 'specificity_gate': specificity_gate,
        'physical_residual_gate': residual_gate,
        'behavioral_rescue_gate': rescue_gate,
    }

any_rescue = any(
    gates['behavioral_rescue_gate'] and gates['specificity_gate']
    for gates in route_gates.values()
)
any_physical_residual = any(
    gates['behavioral_rescue_gate'] and gates['physical_residual_gate']
    for gates in route_gates.values()
)
any_graded = any(
    gates['slope_gate'] and gates['specificity_gate']
    for gates in route_gates.values()
)
if not support_gate:
    PRIMARY_DECISION = 'dose_confirmation_support_insufficient'
elif any_rescue and any_physical_residual:
    PRIMARY_DECISION = 'dose_calibrated_recode_rescues_with_physical_residual'
elif any_rescue:
    PRIMARY_DECISION = 'response_code_steering_rescues_without_residual_advantage'
elif any_graded:
    PRIMARY_DECISION = 'graded_recode_confirmed_without_behavioral_rescue'
else:
    PRIMARY_DECISION = 'recipient_recode_dose_response_not_confirmed'

decision = {
    'primary_decision': PRIMARY_DECISION,
    'selected_alpha': selection,
    'support_gate': support_gate,
    'confirmation_pairs': confirmation_pairs,
    'confirmation_families': confirmation_families,
    'route_gates': route_gates,
    'interpretation_limit': (
        'High-dose target-code steering is not evidence for a physical handoff '
        'unless recoded states outperform target-code-only controls.'
    ),
}
write_json_atomic(ANALYSIS_DIR / 'dose_confirmation_decision.json', decision)
display(decision)


In [ ]:
%%cell_progress 16/17 Audit the split and save the dose-response figure
cell_stage('Checking alpha-zero parity, independent axes, and confirmation isolation')

def centered_list(values):
    values = np.asarray(values, dtype=float)
    return values - values.mean()

zero_errors = []
for pair_id, current in dose_frame[np.isclose(dose_frame['alpha'], 0)].groupby('pair_id'):
    for route in ROUTES:
        rows = current[current['route'] == route].set_index('mode')
        reference = centered_list(rows.loc['recoded', 'candidate_logprobs'])
        for mode in ('random_recoded', 'reverse_target'):
            zero_errors.append(float(np.max(np.abs(
                reference - centered_list(rows.loc[mode, 'candidate_logprobs'])
            ))))
        code_only = centered_list(rows.loc['target_code_only', 'candidate_logprobs'])
        clean = record_frame[
            (record_frame['pair_id'] == pair_id)
            & (record_frame['condition'] == 'clean_target_base')
        ].iloc[0]
        zero_errors.append(float(np.max(np.abs(
            code_only - centered_list(clean['candidate_logprobs'])
        ))))

axis_geometry = []
for pair in ('AB', 'AC', 'BC'):
    axis_geometry.append({
        'pair': pair,
        'calibration_confirmation_cosine': float(torch.sum(
            unit_axis(RESPONSE_AXES[pair]) * unit_axis(CONFIRMATION_AXES[pair])
        )),
    })
axis_geometry_frame = pd.DataFrame(axis_geometry)

audit = {
    'pairs': len(pair_manifest),
    'families': len({row['family_id'] for row in pair_manifest}),
    'discovery_pairs': sum(row['split'] == 'discovery' for row in pair_manifest),
    'confirmation_pairs': sum(row['split'] == 'confirmation' for row in pair_manifest),
    'discovery_confirmation_family_overlap': len(
        {row['family_id'] for row in pair_manifest if row['split'] == 'discovery'}
        & {row['family_id'] for row in pair_manifest if row['split'] == 'confirmation'}
    ),
    'confirmation_rows_used_for_dose_selection': 0,
    'confirmation_families_previously_exposed_at_unit_dose': True,
    'directions_fit_on_current_pairs': False,
    'source_target_labels_always_differ': bool(all(
        row['reference_edited_label'] != row['target_edited_label']
        for row in pair_manifest
    )),
    'maximum_alpha_zero_centered_error': max(zero_errors),
    'minimum_axis_split_cosine': float(
        axis_geometry_frame['calibration_confirmation_cosine'].min()
    ),
    'records_per_pair': int(record_frame.groupby('pair_id').size().min()),
    'finite_primary_measurements': bool(np.isfinite(record_frame[[
        'desired_vs_source_margin', 'desired_vs_error_margin',
        'source_coefficient', 'source_component_fraction',
    ]].to_numpy(dtype=float)).all()),
}
assert audit['discovery_confirmation_family_overlap'] == 0
assert audit['confirmation_rows_used_for_dose_selection'] == 0
assert audit['source_target_labels_always_differ']
assert audit['maximum_alpha_zero_centered_error'] < SMOKE_CENTERED_LOGPROB_TOLERANCE
assert audit['records_per_pair'] == RECORDS_PER_PAIR
assert audit['finite_primary_measurements']
write_json_atomic(ANALYSIS_DIR / 'audit.json', audit)
axis_geometry_frame.to_csv(ANALYSIS_DIR / 'axis_split_geometry.csv', index=False)
display(audit)

fig, axes = plt.subplots(2, 2, figsize=(15, 11))
colors = {
    'recoded': '#2563EB', 'target_code_only': '#16A34A',
    'random_recoded': '#6B7280', 'reverse_target': '#DC2626',
}
labels = {
    'recoded': 'Physical residual + target code',
    'target_code_only': 'Target code only',
    'random_recoded': 'Physical residual + random code',
    'reverse_target': 'Physical residual - target code',
}
for axis, route, title in (
    (axes[0, 0], 'direct', 'A. Direct L55 dose response'),
    (axes[0, 1], 'sequential', 'B. L49 to L55 dose response'),
):
    current = dose_summary[
        (dose_summary['split'] == 'confirmation')
        & (dose_summary['route'] == route)
    ]
    for mode in DOSE_MODES:
        values = current[current['mode'] == mode].sort_values('alpha')
        axis.plot(values['alpha'], values['desired_margin'], marker='o',
                  color=colors[mode], label=labels[mode])
    axis.axhline(0, color='black', linewidth=1)
    axis.axvline(selection[route], color='#D97706', linestyle='--',
                 label='Discovery-selected dose')
    axis.set_xlabel('Target-code dose multiplier')
    axis.set_ylabel('Target minus source label margin')
    axis.set_title(title)
    axis.legend(frameon=False, fontsize=8)

x = np.arange(len(ROUTES))
width = 0.19
for index, mode in enumerate(DOSE_MODES):
    values = [
        float(selected_frame[
            (selected_frame['route'] == route)
            & (selected_frame['mode'] == mode)
        ]['success_rate'].iloc[0]) for route in ROUTES
    ]
    axes[1, 0].bar(x + (index - 1.5) * width, values, width,
                   color=colors[mode], label=labels[mode])
axes[1, 0].set_xticks(x, ['Direct L55', 'Sequential L49 to L55'])
axes[1, 0].set_ylim(0, 1)
axes[1, 0].set_ylabel('Confirmation target-answer rate')
axes[1, 0].set_title('C. Behavioral rescue at discovery-selected doses')
axes[1, 0].legend(frameon=False, fontsize=8)

forest = contrast_frame[
    (contrast_frame['metric'] == 'margin')
    & (contrast_frame['contrast'].isin([
        'recoded_vs_random_recoded', 'recoded_vs_reverse_target',
        'recoded_vs_target_code_only', 'recoded_vs_baseline',
    ]))
].copy()
forest['label'] = forest['route'] + ': ' + forest['contrast'].str.replace('_', ' ')
y = np.arange(len(forest))
axes[1, 1].errorbar(
    forest['estimate'], y,
    xerr=np.vstack([
        forest['estimate'] - forest['ci_low'],
        forest['ci_high'] - forest['estimate'],
    ]), fmt='o', color='#111827', capsize=4,
)
axes[1, 1].axvline(0, color='#9CA3AF', linestyle='--')
axes[1, 1].set_yticks(y, forest['label'])
axes[1, 1].set_title('D. Held-out family-bootstrap contrasts')

fig.suptitle('Dose-calibrated recipient response-code recoding', fontsize=16)
fig.tight_layout()
figure_path = FIGURE_DIR / 'recipient_recode_dose_response.png'
fig.savefig(figure_path, dpi=220, bbox_inches='tight')
plt.show()
print('Saved:', figure_path)


In [ ]:
%%cell_progress 17/17 Write the confirmatory decision and report
cell_stage('Saving paper-facing tables and the conservative claim')

def interval(route, contrast, metric):
    row = lookup(route, contrast, metric)
    return f"{row['estimate']:.4f} [{row['ci_low']:.4f}, {row['ci_high']:.4f}]"

lines = [
    '# Recipient Recode Dose-Response Confirmation', '',
    f'Primary decision: {PRIMARY_DECISION}', '',
    '## Why this experiment was necessary', '',
    (
        'The unit-dose intervention moved every continuous contrast in the '
        'predicted direction but did not change any discrete answer. This '
        'experiment tests whether that was a subthreshold effect without '
        'choosing a favorable strength on the confirmation examples.'
    ), '',
    '## Discovery-selected doses', '',
    f'- Direct L55: {selection["direct"]}',
    f'- Sequential L49 to L55: {selection["sequential"]}', '',
    '## Held-out direct L55 contrasts', '',
    f'- Recoded versus random margin: {interval("direct", "recoded_vs_random_recoded", "margin")}',
    f'- Recoded versus reverse margin: {interval("direct", "recoded_vs_reverse_target", "margin")}',
    f'- Recoded versus target-code-only margin: {interval("direct", "recoded_vs_target_code_only", "margin")}',
    f'- Recoded versus raw margin: {interval("direct", "recoded_vs_baseline", "margin")}',
    f'- Selected-dose target success: {rate("direct", "recoded"):.3f}', '',
    '## Held-out sequential contrasts', '',
    f'- Recoded versus random margin: {interval("sequential", "recoded_vs_random_recoded", "margin")}',
    f'- Recoded versus reverse margin: {interval("sequential", "recoded_vs_reverse_target", "margin")}',
    f'- Recoded versus target-code-only margin: {interval("sequential", "recoded_vs_target_code_only", "margin")}',
    f'- Recoded versus unmodified L49 transport: {interval("sequential", "recoded_vs_baseline", "margin")}',
    f'- Selected-dose target success: {rate("sequential", "recoded"):.3f}', '',
    '## Interpretation boundary', '',
    (
        'A high-dose answer flip alone only establishes response-code '
        'controllability. Evidence that the transported physical update '
        'contributes requires the recoded condition to outperform the '
        'target-code-only condition on held-out confirmation families.'
    ),
    (
        'The parent confirmation families are held out from the current dose '
        'selection, but they were already observed at the unit dose. This is '
        'therefore a prospective dose test on reused support, not a new-cohort '
        'independent replication.'
    ), '',
]
report_path = ANALYSIS_DIR / 'DOSE_RESPONSE_CONFIRMATION_REPORT.md'
report_path.write_text('\n'.join(lines), encoding='utf-8')

run_complete = {
    'schema_version': 'recipient-recode-dose-response-run-1.0',
    'primary_decision': PRIMARY_DECISION,
    'model_id': MODEL_ID, 'revision': MODEL_REVISION,
    'selected_alpha': selection,
    'confirmation_pairs': confirmation_pairs,
    'confirmation_families': confirmation_families,
    'route_gates': route_gates,
    'protocol_sha256': sha256_file(PROTOCOL_LOCK_PATH),
    'pair_manifest_sha256': sha256_file(PAIR_MANIFEST_PATH),
    'axis_tensor_sha256': sha256_file(AXIS_TENSOR_PATH),
    'capture_sha256': CAPTURE_SHA256,
    'dose_curves': str(ANALYSIS_DIR / 'dose_curves.csv'),
    'selected_rates': str(ANALYSIS_DIR / 'selected_dose_confirmation_rates.csv'),
    'contrasts': str(ANALYSIS_DIR / 'confirmation_contrasts.csv'),
    'decision': str(ANALYSIS_DIR / 'dose_confirmation_decision.json'),
    'audit': str(ANALYSIS_DIR / 'audit.json'),
    'figure': str(FIGURE_DIR / 'recipient_recode_dose_response.png'),
    'report': str(report_path),
    'completed_at_utc': datetime.now(timezone.utc).isoformat(),
}
write_json_atomic(OUTPUT_ROOT / 'RUN_COMPLETE.json', run_complete)
display(run_complete)
print(report_path.read_text(encoding='utf-8'))
